# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
from IPython.display import display

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("DuckDB + Hugging Face connection ready.")

DuckDB + Hugging Face connection ready.


## 1. Unit of analysis + time window

My lane is **Refresh / Content Opportunity Scoring**.

The raw `fact_content_daily_performance` table has one row per
`report_date × client_hash_id × content_hash_id`.

For this analysis, I aggregate the daily data so that one modeling row
represents one pseudonymized content page summarized for a 14-day review cycle.

I use two tables:

- `fact_content_daily_performance` for observed GSC performance signals.
- `dim_content` for content metadata and joins.

I develop the contract on March 2026, a mid-panel month.

- Previous / feature window: `2026-03-01` through `2026-03-14`
- Current review window: `2026-03-15` through `2026-03-28`
- Decision date: `2026-03-28`

The previous window provides the point-in-time-safe search-performance
features. `content_age_days` is computed from metadata already known by the
2026-03-28 decision date. The current window is used to construct the
provisional review proxy.

The output is a ranked human-review queue. A page ranked highly means that
it deserves review based on observed signals; it does not mean that the page
definitely needs a refresh or that a refresh will improve performance.

## 2. Fields: feature / label / context / excluded

### Label / proxy

`needs_review_proxy` is a provisional decision-support proxy for whether a
page deserves human review.

A page is positive when at least one of these review signals is present:

- **Declining with demand:** previous 14-day impressions are at least 100,
  the current window has at least one GSC-available day, and current 14-day
  impressions fell by at least 20%.
- **Low CTR while visible:** current 14-day impressions are at least 100 and
  CTR is at least 25% below the median CTR for pages in the same position tier.

This proxy does not mean that a page definitely needs a refresh or that a
refresh will improve performance.

GA4 engagement was evaluated during contract design, but its coverage was too
sparse to use as a main model feature or core proxy trigger in this March
slice. `page_one_decay_risk` is kept as an additional priority modifier rather
than a separate proxy trigger.

### Features

I use three point-in-time-safe model features:

1. `prev_14d_avg_position` — weighted average search position from
   2026-03-01 through 2026-03-14. It is knowable at the decision moment
   because it uses only observations from the previous window.

2. `prev_14d_ctr` — clicks divided by impressions in the previous
   14-day window. It is knowable at the decision moment because both
   clicks and impressions were observed before the current review window.

3. `content_age_days` — days between `content_created_date` and the
   2026-03-28 decision date. Creation date is historical metadata already
   known by the decision moment.

`word_count` and `days_since_last_update` were considered but are not used
as model features in this contract. The available `dim_content` table is a
current snapshot, so for pages updated after the decision date I cannot
reconstruct those metadata values exactly as they were on 2026-03-28.
Using the current values could introduce future information.

### Context fields

`client_hash_id`, `content_hash_id`, and `report_date` are used for joining,
grouping, defining time windows, and validation. They are not predictive
features.

`gsc_data_available` and `ga4_data_available` are availability/context flags.
They are used to avoid interpreting unavailable measurements as real zeros.

### Deliberately excluded

`current_14d_ctr` is deliberately excluded from the honest model feature set
because it contributes directly to the `low_ctr_while_visible` proxy rule.
Using it as a model feature would partially reveal the answer and create a
circular result.

The proxy flags themselves, including `declining_with_demand`,
`low_ctr_while_visible`, `page_one_decay_risk`, and `needs_review_proxy`, are
also not used as honest model features.

## 3. Verify it with queries (grain, counts, missing values, windows)

I verify the contract on the March 2026 mid-panel slice before modeling.

The three warehouse verification queries check:

1. the raw daily-table grain,
2. the March slice size and date span,
3. GA4 availability using an explicit `IS TRUE` filter.

After those checks, I construct the three point-in-time-safe features,
the provisional review proxy, and the leakage demonstration.

In [2]:
# Query 1 — Verify the raw daily-table grain.
# Expected grain: one row per report_date + client + content page.

grain_check = con.sql(f"""
WITH grain_groups AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM {MARCH_DAILY}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
)
SELECT
    COUNT(*) AS unique_grain_keys,
    SUM(rows_per_key) AS source_rows,
    SUM(CASE WHEN rows_per_key > 1 THEN 1 ELSE 0 END) AS duplicate_key_groups,
    MAX(rows_per_key) AS max_rows_per_key
FROM grain_groups
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,unique_grain_keys,source_rows,duplicate_key_groups,max_rows_per_key
0,9841378,9841378.0,0.0,1


In [3]:
# Query 2 — Verify the March 2026 slice size and date span.

slice_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_hash_id) AS unique_content_pages,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {MARCH_DAILY}
""").df()

display(slice_check)

,row_count,unique_content_pages,unique_clients,min_report_date,max_report_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [4]:
# Query 3 — Check GA4 availability using IS TRUE
# and show how many rows survive the availability filter.

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
    SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_unavailable_or_null_rows,
    ROUND(
        100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS pct_rows_with_ga4
FROM {MARCH_DAILY}
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_unavailable_or_null_rows,pct_rows_with_ga4
0,9841378,413966.0,9427412.0,4.21


In [5]:
# Build the page-level feature frame and provisional review proxy.
#
# Decision date:   2026-03-28
# Previous window: 2026-03-01 to 2026-03-14
# Current window:  2026-03-15 to 2026-03-28
#
# Honest model features:
#   1. prev_14d_avg_position
#   2. prev_14d_ctr
#   3. content_age_days

feature_frame = con.sql(f"""
WITH page_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- =========================
        -- Previous 14-day window
        -- =========================

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS prev_14d_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS prev_14d_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_sum_position, 0)
                ELSE 0
            END
        ) AS prev_14d_sum_position,

        -- =========================
        -- Current 14-day window
        -- Used to define the proxy,
        -- NOT as honest model features
        -- =========================

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-15'
                                     AND DATE '2026-03-28'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS current_14d_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-15'
                                     AND DATE '2026-03-28'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS current_14d_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-15'
                                     AND DATE '2026-03-28'
                     AND gsc_data_available IS TRUE
                THEN COALESCE(gsc_sum_position, 0)
                ELSE 0
            END
        ) AS current_14d_sum_position,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-15'
                                     AND DATE '2026-03-28'
                     AND gsc_data_available IS TRUE
                THEN 1
                ELSE 0
            END
        ) AS current_gsc_available_days

    FROM {MARCH_DAILY}

    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-28'

    GROUP BY
        client_hash_id,
        content_hash_id
),

rates AS (
    SELECT
        *,

        -- Honest feature 1
        CASE
            WHEN prev_14d_impressions > 0
            THEN prev_14d_sum_position / prev_14d_impressions
        END AS prev_14d_avg_position,

        -- Honest feature 2
        CASE
            WHEN prev_14d_impressions > 0
            THEN 100.0 * prev_14d_clicks / prev_14d_impressions
        END AS prev_14d_ctr,

        -- Current-window values used only to construct the proxy
        CASE
            WHEN current_14d_impressions > 0
            THEN current_14d_sum_position / current_14d_impressions
        END AS current_14d_avg_position,

        CASE
            WHEN current_14d_impressions > 0
            THEN 100.0 * current_14d_clicks / current_14d_impressions
        END AS current_14d_ctr

    FROM page_windows
),

with_tiers AS (
    SELECT
        *,

        CASE
            WHEN current_14d_avg_position IS NULL THEN 'no_data'
            WHEN current_14d_avg_position <= 3 THEN 'top_3'
            WHEN current_14d_avg_position <= 10 THEN 'page_1'
            WHEN current_14d_avg_position <= 20 THEN 'striking'
            WHEN current_14d_avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS current_position_tier

    FROM rates
),

tier_medians AS (
    SELECT
        current_position_tier,
        MEDIAN(current_14d_ctr) AS tier_median_ctr

    FROM with_tiers

    WHERE current_14d_impressions >= 100
      AND current_14d_avg_position IS NOT NULL
      AND current_14d_ctr IS NOT NULL
      AND current_position_tier <> 'no_data'

    GROUP BY current_position_tier
),

joined AS (
    SELECT
        w.*,
        t.tier_median_ctr,

        -- Honest feature 3
        CASE
            WHEN c.content_created_date <= DATE '2026-03-28'
            THEN DATE_DIFF(
                'day',
                c.content_created_date,
                DATE '2026-03-28'
            )
        END AS content_age_days

    FROM with_tiers AS w

    LEFT JOIN tier_medians AS t
        USING (current_position_tier)

    LEFT JOIN {DIM_CONTENT} AS c
        ON w.content_hash_id = c.content_hash_id
),

signals AS (
    SELECT
        *,

        -- Core review signal 1
        CASE
            WHEN prev_14d_impressions >= 100
             AND current_gsc_available_days > 0
             AND current_14d_impressions
                 <= 0.80 * prev_14d_impressions
            THEN 1
            ELSE 0
        END AS declining_with_demand,

        -- Core review signal 2
        CASE
            WHEN current_14d_impressions >= 100
             AND current_14d_avg_position IS NOT NULL
             AND tier_median_ctr > 0
             AND current_14d_ctr <= 0.75 * tier_median_ctr
            THEN 1
            ELSE 0
        END AS low_ctr_while_visible,

        -- Priority modifier only
        CASE
            WHEN prev_14d_impressions >= 100
             AND prev_14d_avg_position > 0
             AND prev_14d_avg_position <= 10
             AND current_14d_avg_position
                 >= prev_14d_avg_position + 2
            THEN 1
            ELSE 0
        END AS page_one_decay_risk

    FROM joined
)

SELECT
    *,

    CASE
        WHEN declining_with_demand = 1
          OR low_ctr_while_visible = 1
        THEN 1
        ELSE 0
    END AS needs_review_proxy

FROM signals
""").df()

print("Feature frame shape:", feature_frame.shape)

display(
    feature_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "prev_14d_avg_position",
            "prev_14d_ctr",
            "content_age_days",
            "declining_with_demand",
            "low_ctr_while_visible",
            "page_one_decay_risk",
            "needs_review_proxy",
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (326194, 20)


,client_hash_id,content_hash_id,prev_14d_avg_position,prev_14d_ctr,content_age_days,declining_with_demand,low_ctr_while_visible,page_one_decay_risk,needs_review_proxy
0,client_1a730cb2640a1abf,content_3b2dc188104260ad,12.666149,0.155280,66,0,0,0,0
1,client_1a730cb2640a1abf,content_6c1629113b004f4d,3.324916,0.673401,66,1,0,1,1
2,client_1a730cb2640a1abf,content_13ef113e2713d5d8,0.000000,0.000000,60,0,1,0,1
3,client_1a730cb2640a1abf,content_ab17bfe2f5594274,7.735294,0.000000,58,0,0,0,0
4,client_1a730cb2640a1abf,content_a705d80c33152cd6,7.921739,0.000000,58,1,1,1,1


In [6]:
feature_summary = pd.DataFrame({
    "metric": [
        "Feature-frame rows",
        "Needs-review positives",
        "Needs-review negatives",
        "Declining-with-demand pages",
        "Low-CTR-visible pages",
        "Page-one-decay pages",
    ],
    "value": [
        len(feature_frame),
        int(feature_frame["needs_review_proxy"].sum()),
        int((feature_frame["needs_review_proxy"] == 0).sum()),
        int(feature_frame["declining_with_demand"].sum()),
        int(feature_frame["low_ctr_while_visible"].sum()),
        int(feature_frame["page_one_decay_risk"].sum()),
    ],
})

display(feature_summary)

print(
    "Proxy positive rate:",
    f'{feature_frame["needs_review_proxy"].mean() * 100:.2f}%'
)

,metric,value
0,Feature-frame rows,326194
1,Needs-review positives,46884
2,Needs-review negatives,279310
3,Declining-with-demand pages,25406
4,Low-CTR-visible pages,29547
5,Page-one-decay pages,11472


Proxy positive rate: 14.37%


In [7]:
feature_coverage = pd.DataFrame({
    "feature": [
        "prev_14d_avg_position",
        "prev_14d_ctr",
        "content_age_days",
    ],
    "non_missing": [
        feature_frame["prev_14d_avg_position"].notna().sum(),
        feature_frame["prev_14d_ctr"].notna().sum(),
        feature_frame["content_age_days"].notna().sum(),
    ],
})

feature_coverage["coverage_pct"] = (
    100.0 * feature_coverage["non_missing"] / len(feature_frame)
).round(2)

display(feature_coverage)

,feature,non_missing,coverage_pct
0,prev_14d_avg_position,150444,46.12
1,prev_14d_ctr,150444,46.12
2,content_age_days,324952,99.62


In [8]:
# Define the modeling cohort:
# keep pages with all three honest features available.

complete_feature_mask = (
    feature_frame["prev_14d_avg_position"].notna()
    & feature_frame["prev_14d_ctr"].notna()
    & feature_frame["content_age_days"].notna()
)

complete_frame = feature_frame.loc[
    complete_feature_mask
].copy()

coverage_decision = pd.DataFrame({
    "metric": [
        "All feature-frame pages",
        "Pages with all 3 features",
        "Percent of pages retained",
        "All proxy positives",
        "Proxy positives retained",
        "Percent of positives retained",
        "Positive rate in complete-feature cohort",
        "Positive rate among excluded pages",
    ],
    "value": [
        len(feature_frame),

        len(complete_frame),

        round(
            100 * len(complete_frame)
            / len(feature_frame),
            2
        ),

        int(
            feature_frame[
                "needs_review_proxy"
            ].sum()
        ),

        int(
            complete_frame[
                "needs_review_proxy"
            ].sum()
        ),

        round(
            100
            * complete_frame[
                "needs_review_proxy"
            ].sum()
            / feature_frame[
                "needs_review_proxy"
            ].sum(),
            2
        ),

        round(
            100
            * complete_frame[
                "needs_review_proxy"
            ].mean(),
            2
        ),

        round(
            100
            * feature_frame.loc[
                ~complete_feature_mask,
                "needs_review_proxy"
            ].mean(),
            2
        ),
    ],
})

display(coverage_decision)

,metric,value
0,All feature-frame pages,326194.00
1,Pages with all 3 features,150444.00
2,Percent of pages retained,46.12
3,All proxy positives,46884.00
4,Proxy positives retained,45874.00
5,Percent of positives retained,97.85
6,Positive rate in complete-feature cohort,30.49
7,Positive rate among excluded pages,0.57


### Leakage trap

To demonstrate leakage, I first train a quick model using only the three
point-in-time-safe features.

I then intentionally add one label-derived feature,
`leaky_proxy_copy = needs_review_proxy`.

Because this column directly contains the answer, model performance should
jump toward perfect. I then remove the leaky feature and keep only the honest
model result.

In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# --------------------------------------------------
# Modeling data
# --------------------------------------------------

model_frame = complete_frame[
    [
        "client_hash_id",
        "prev_14d_avg_position",
        "prev_14d_ctr",
        "content_age_days",
        "needs_review_proxy",
    ]
].copy()

HONEST_FEATURES = [
    "prev_14d_avg_position",
    "prev_14d_ctr",
    "content_age_days",
]

X = model_frame[HONEST_FEATURES]
y = model_frame["needs_review_proxy"]
groups = model_frame["client_hash_id"]

# --------------------------------------------------
# Client-holdout split
# Whole clients stay either in train or test.
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())
print(
    "Test positive rate:",
    f"{y_test.mean() * 100:.2f}%"
)

# --------------------------------------------------
# Honest quick model
# --------------------------------------------------

honest_model = Pipeline([
    ("scale", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

honest_model.fit(
    X_train,
    y_train,
)

honest_prob = honest_model.predict_proba(
    X_test
)[:, 1]

honest_ap = average_precision_score(
    y_test,
    honest_prob,
)

print(
    "\nHonest Average Precision:",
    round(honest_ap, 4)
)

Train rows: 137599
Test rows: 12845
Train clients: 35
Test clients: 9
Test positive rate: 26.25%

Honest Average Precision: 0.2679


In [10]:
# --------------------------------------------------
# INTENTIONAL LEAKAGE TRAP
# --------------------------------------------------
# This feature is deliberately copied from the target.
# It must never be used in the honest model.

leaky_frame = model_frame.copy()

leaky_frame["leaky_proxy_copy"] = (
    leaky_frame["needs_review_proxy"]
)

LEAKY_FEATURES = HONEST_FEATURES + [
    "leaky_proxy_copy"
]

X_leaky = leaky_frame[LEAKY_FEATURES]

# Use exactly the same client-holdout split
X_leaky_train = X_leaky.iloc[train_idx]
X_leaky_test = X_leaky.iloc[test_idx]

leaky_model = Pipeline([
    ("scale", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

leaky_model.fit(
    X_leaky_train,
    y_train,
)

leaky_prob = leaky_model.predict_proba(
    X_leaky_test
)[:, 1]

leaky_ap = average_precision_score(
    y_test,
    leaky_prob,
)

print(
    "Honest Average Precision:",
    round(honest_ap, 4)
)

print(
    "Leaky Average Precision: ",
    round(leaky_ap, 4)
)

print(
    "Score increase:",
    round(leaky_ap - honest_ap, 4)
)

Honest Average Precision: 0.2679
Leaky Average Precision:  1.0
Score increase: 0.7321


#### Leakage-check result

The honest logistic-regression model achieved an Average Precision of
`0.2679` on held-out clients.

The positive rate in the held-out test set was `26.25%`, so the honest model
performed only slightly above a no-skill ranking baseline. This suggests that
the three point-in-time-safe features contain limited predictive signal for
the current review proxy.

I then intentionally added `leaky_proxy_copy`, which was an exact copy of
`needs_review_proxy`. Average Precision jumped from `0.2679` to `1.0000`.

This perfect result is not evidence of a better model. It demonstrates target
leakage: the model was given the answer as an input feature.

The leaky feature is removed immediately, and only the honest result
(`0.2679`) is retained.

In [11]:
# Remove the intentionally leaky feature.

leaky_frame = leaky_frame.drop(
    columns=["leaky_proxy_copy"]
)

assert "leaky_proxy_copy" not in leaky_frame.columns

print("Leaky feature removed.")
print(
    "Honest result retained:",
    round(honest_ap, 4)
)

Leaky feature removed.
Honest result retained: 0.2679


## 4. Data limits

The main limitation is that `needs_review_proxy` is a rule-based review
proxy, not ground truth that a page truly needs a refresh or that refreshing
it will improve performance.

The modeling cohort is limited to pages with observable previous-window GSC
history. Pages without enough prior search evidence cannot have reliable
previous CTR or average-position features. The retained modeling cohort keeps
97.85% of proxy-positive pages, but the model should not be interpreted as
covering every page in the inventory.

GA4 engagement was evaluated during contract design, but coverage was sparse
in this March slice. Engagement therefore is not used as a main model feature
or as a core proxy trigger.

Mutable metadata such as current `word_count` and
`days_since_last_update` was excluded from the honest model because the
available content dimension is a current snapshot and does not provide a
historical version of those values as of the 2026-03-28 decision date.

The honest logistic-regression result was only slightly above the held-out
no-skill reference (`0.2679` Average Precision versus a `0.2625` positive
rate). This suggests that the three safe features contain limited predictive
signal for the current proxy, so the model result should be treated as a
leakage check and early diagnostic rather than strong evidence of useful
ranking performance.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.